# Pandas Function APIs in PySpark 3

PySpark 3.0+ ships three **pandas function APIs** that process entire
`pd.DataFrame` batches instead of individual rows.  They combine the
expressiveness of pandas with Spark's distributed execution:

| API | Scope | Typical Use |
|-----|-------|-------------|
| `mapInPandas` | per-partition | batch-wise row transforms |
| `applyInPandas` | per-group | split-apply-combine analytics |
| `cogroup().applyInPandas` | per-key across two DFs | cross-dataset joins/logic |

This notebook walks through each API with runnable examples.

In [ ]:
import os
os.environ.setdefault("PYARROW_IGNORE_TIMEZONE", "1")

from typing import Iterator

import pandas as pd
from pyspark.sql import functions as F

from spp.session import create_spark_session

spark = create_spark_session("notebook-function-apis")
print(f"Spark {spark.version}")

## 1. mapInPandas — Batch-wise Transformation

`mapInPandas` processes each **partition** of a Spark DataFrame as an
iterator of `pd.DataFrame` batches.  Your function receives the iterator,
transforms each batch with familiar pandas operations, and yields the
results back.

Key points:
- Available since **Spark 3.0.0**
- Operates on the full row (all columns visible)
- The output schema must be declared explicitly
- Great for row-wise transforms that benefit from vectorised pandas methods

In [ ]:
df = spark.createDataFrame(
    [(1, 21), (2, 30), (3, 25), (4, 35)],
    ["id", "age"],
)


def add_double_age(iterator: Iterator[pd.DataFrame]) -> Iterator[pd.DataFrame]:
    """Add a ``double_age`` column equal to ``age * 2``."""
    for pdf in iterator:
        pdf["double_age"] = pdf["age"] * 2
        yield pdf


result = df.mapInPandas(
    add_double_age,
    schema="id: bigint, age: bigint, double_age: bigint",
)
result.show()

## 2. applyInPandas — Grouped Map

`applyInPandas` implements the **split-apply-combine** pattern:

1. **Split** — Spark groups the DataFrame by one or more keys
2. **Apply** — your function receives each group as a `pd.DataFrame`
3. **Combine** — Spark reassembles the results into a single DataFrame

This is ideal for per-group statistics (z-scores, rolling windows, etc.)
that are easy to express in pandas but hard with built-in Spark functions.

In [ ]:
df = spark.createDataFrame(
    [(1, 1.0), (1, 2.0), (1, 3.0), (2, 4.0), (2, 5.0), (2, 10.0)],
    ["group_id", "value"],
)


def normalize_group(pdf: pd.DataFrame) -> pd.DataFrame:
    """Z-score normalise ``value`` within each group."""
    pdf = pdf.copy()
    std = pdf["value"].std() or 1.0
    pdf["normalized"] = (pdf["value"] - pdf["value"].mean()) / std
    return pdf


df.groupBy("group_id").applyInPandas(
    normalize_group,
    schema="group_id: long, value: double, normalized: double",
).show()

### Group Key Access

Your function can optionally accept `(key, pdf)` instead of just `pdf`.
The key is a **tuple** of the grouping column values — useful for
including the group identifier in aggregated output.

In [ ]:
def mean_with_key(key: tuple, pdf: pd.DataFrame) -> pd.DataFrame:
    """Return the group mean, including the group key."""
    return pd.DataFrame([key + (pdf["value"].mean(),)])


df.groupBy("group_id").applyInPandas(
    mean_with_key,
    schema="group_id: long, mean_value: double",
).show()

## 3. cogroup().applyInPandas — Cogrouped Map

`cogroup` pairs two DataFrames by a shared key and hands both groups
to your function as two `pd.DataFrame` arguments.  This lets you
implement custom join logic, e.g. left-merge, asof-join, or any
operation that needs simultaneous access to both sides.

In [ ]:
df1 = spark.createDataFrame(
    [(1, 1.0), (1, 2.0), (2, 3.0), (2, 4.0)],
    ["id", "value1"],
)
df2 = spark.createDataFrame(
    [(1, "A"), (2, "B"), (2, "C")],
    ["id", "value2"],
)


def combine_groups(pdf1: pd.DataFrame, pdf2: pd.DataFrame) -> pd.DataFrame:
    """Left-merge two grouped DataFrames on ``id``."""
    return pdf1.merge(pdf2, on="id", how="left")


df1.groupBy("id").cogroup(df2.groupBy("id")).applyInPandas(
    combine_groups,
    schema="id: long, value1: double, value2: string",
).show()

## Performance Configuration

Arrow controls the batch size exchanged between the JVM and Python.
Tuning `maxRecordsPerBatch` can improve throughput for large partitions
or reduce memory pressure for wide schemas.

In [ ]:
spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "5000")

print("Arrow enabled:",
      spark.conf.get("spark.sql.execution.arrow.pyspark.enabled"))
print("Max records per batch:",
      spark.conf.get("spark.sql.execution.arrow.maxRecordsPerBatch"))

## When to Use Which Method

| Method | Input | Best For |
|--------|-------|----------|
| `mapInPandas` | Iterator of batches (whole partition) | Row-wise transforms, filtering, adding columns |
| `applyInPandas` | One `pd.DataFrame` per group | Per-group analytics: z-scores, rolling stats, resampling |
| `cogroup().applyInPandas` | Two `pd.DataFrame`s per key | Custom joins, cross-dataset comparisons |

**Tips:**
- All three require an explicit output **schema** string.
- Arrow is used under the hood — keep it enabled for best performance.
- Prefer `mapInPandas` when you don't need grouping; it avoids a shuffle.

In [ ]:
spark.stop()